# 02 — PSF Diagnostics

The PSF model is the foundation of shape measurement. If the PSF model
is wrong, every galaxy shape is wrong. This notebook validates the PSF
by comparing model predictions to actual star measurements.

**Key diagnostics:**
1. PSF size residuals: $(T_\star - T_{\text{model}}) / T_\star$
2. PSF ellipticity residuals: $e_\star - e_{\text{model}}$
3. Spatial patterns in the residuals
4. ρ-statistics (correlation functions of PSF residuals)

**Run this on the Rubin Science Platform (RSP).**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

from lsst.daf.butler import Butler
import lsst.geom as geom

## 1. Load Stars from a Single Visit

PSF diagnostics are best done on individual visits (not coadds),
since each visit has its own PSF. We'll use a `src` catalog
and select stars.

In [ ]:
butler = Butler('dp02', collections='2.2i/runs/DP0.2')

# Pick a visit and detector
# (visit numbers depend on the DP0.2 dataset — adjust as needed)
visit = 192350
detector = 75

# Get the source catalog for this visit/detector
src = butler.get('src', visit=visit, detector=detector)

# Also get the calibrated exposure (for the PSF model)
calexp = butler.get('calexp', visit=visit, detector=detector)

print(f"Sources in this CCD: {len(src)}")
print(f"PSF model type: {type(calexp.getPsf()).__name__}")

In [ ]:
# Select stars: use the calib_psf_used flag
# These are the stars the PSF model was fit to
psf_stars = src[src['calib_psf_used']].copy(deep=True)

# Also get the reserved stars (not used for fitting — independent validation)
if 'calib_psf_reserved' in src.schema.getNames():
    reserved_stars = src[src['calib_psf_reserved']].copy(deep=True)
    print(f"PSF fitting stars: {len(psf_stars)}")
    print(f"Reserved stars (for validation): {len(reserved_stars)}")
    validation_set = reserved_stars
else:
    print(f"PSF stars: {len(psf_stars)} (no reserved set available)")
    validation_set = psf_stars

## 2. Compare Star Shapes to PSF Model Predictions

In [ ]:
# Extract measured star moments and PSF model moments
# SdssShape gives adaptive moments: Ixx, Iyy, Ixy

star_Ixx = validation_set['base_SdssShape_xx']
star_Iyy = validation_set['base_SdssShape_yy']
star_Ixy = validation_set['base_SdssShape_xy']

psf_Ixx = validation_set['base_SdssShape_psf_xx']
psf_Iyy = validation_set['base_SdssShape_psf_yy']
psf_Ixy = validation_set['base_SdssShape_psf_xy']

# Trace (size²)
T_star = star_Ixx + star_Iyy
T_psf = psf_Ixx + psf_Iyy

# Fractional size residual
delta_T_frac = (T_star - T_psf) / T_star

# Ellipticity residuals
e1_star = (star_Ixx - star_Iyy) / T_star
e2_star = 2 * star_Ixy / T_star
e1_psf = (psf_Ixx - psf_Iyy) / T_psf
e2_psf = 2 * psf_Ixy / T_psf

delta_e1 = e1_star - e1_psf
delta_e2 = e2_star - e2_psf

# Positions
x = validation_set['base_SdssCentroid_x']
y = validation_set['base_SdssCentroid_y']

print(f"Fractional size residual: mean={np.mean(delta_T_frac):.5f}, "
      f"std={np.std(delta_T_frac):.5f}")
print(f"e1 residual: mean={np.mean(delta_e1):.6f}, std={np.std(delta_e1):.5f}")
print(f"e2 residual: mean={np.mean(delta_e2):.6f}, std={np.std(delta_e2):.5f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Fractional size residual
ax = axes[0]
ax.hist(delta_T_frac, bins=40, range=(-0.05, 0.05),
        color='steelblue', edgecolor='white')
ax.axvline(0, color='red', ls='--', lw=1.5)
ax.axvline(np.mean(delta_T_frac), color='orange', lw=2,
           label=f'mean={np.mean(delta_T_frac):.5f}')
ax.set_xlabel('$(T_\star - T_{\\rm PSF}) / T_\star$', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Fractional size residual', fontsize=13)
ax.legend()
# LSST requirement
ax.axvline(0.001, color='gray', ls=':', alpha=0.5)
ax.axvline(-0.001, color='gray', ls=':', alpha=0.5, label='LSST req: ±0.001')

# e1 residual
ax = axes[1]
ax.hist(delta_e1, bins=40, range=(-0.02, 0.02),
        color='forestgreen', edgecolor='white')
ax.axvline(0, color='red', ls='--', lw=1.5)
ax.axvline(np.mean(delta_e1), color='orange', lw=2,
           label=f'mean={np.mean(delta_e1):.6f}')
ax.set_xlabel('$e_{1,\star} - e_{1,\\rm PSF}$', fontsize=12)
ax.set_title('$e_1$ residual', fontsize=13)
ax.legend()

# e2 residual
ax = axes[2]
ax.hist(delta_e2, bins=40, range=(-0.02, 0.02),
        color='darkorange', edgecolor='white')
ax.axvline(0, color='red', ls='--', lw=1.5)
ax.axvline(np.mean(delta_e2), color='blue', lw=2,
           label=f'mean={np.mean(delta_e2):.6f}')
ax.set_xlabel('$e_{2,\star} - e_{2,\\rm PSF}$', fontsize=12)
ax.set_title('$e_2$ residual', fontsize=13)
ax.legend()

plt.suptitle(f'PSF Model Residuals (visit={visit}, det={detector}, '
             f'N={len(validation_set)} stars)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../figures/psf_residuals_hist.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Spatial Pattern of PSF Residuals

Residuals should be **spatially random**. Coherent patterns mean the
PSF model is missing real spatial variation, which directly creates
spurious shear correlations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Whisker plot: size residuals
ax = axes[0]
sc = ax.scatter(x, y, c=delta_T_frac, cmap='RdBu_r', s=30,
                vmin=-0.02, vmax=0.02, edgecolors='gray', linewidths=0.3)
plt.colorbar(sc, ax=ax, label='$\\Delta T / T$')
ax.set_xlabel('x (pixels)'); ax.set_ylabel('y (pixels)')
ax.set_title('Size residual across CCD')
ax.set_aspect('equal')

# Whisker plot: e1 residuals
ax = axes[1]
sc = ax.scatter(x, y, c=delta_e1, cmap='RdBu_r', s=30,
                vmin=-0.01, vmax=0.01, edgecolors='gray', linewidths=0.3)
plt.colorbar(sc, ax=ax, label='$\\Delta e_1$')
ax.set_xlabel('x (pixels)'); ax.set_ylabel('y (pixels)')
ax.set_title('$e_1$ residual across CCD')
ax.set_aspect('equal')

# Whisker plot: ellipticity whiskers
ax = axes[2]
scale = 50  # whisker scale
ax.quiver(x, y, delta_e1 * scale, delta_e2 * scale,
          np.sqrt(delta_e1**2 + delta_e2**2),
          cmap='hot_r', scale=3, headwidth=0, headlength=0, headaxislength=0)
ax.set_xlabel('x (pixels)'); ax.set_ylabel('y (pixels)')
ax.set_title('PSF ellipticity residual whiskers')
ax.set_aspect('equal')

plt.suptitle('Spatial Pattern of PSF Residuals', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../figures/psf_residuals_spatial.png', dpi=150, bbox_inches='tight')
plt.show()

print("Look for:")
print("  - Smooth gradients → PSF polynomial order too low")
print("  - Ring patterns → optical aberrations not captured")
print("  - Edge effects → poor interpolation near CCD boundaries")
print("  - Random scatter → good PSF model!")

## 4. PSF Ellipticity Pattern (Model, not Residuals)

It's also useful to see the **PSF model itself** — the pattern
of PSF ellipticity across the CCD shows the optics signature.

In [ ]:
# Evaluate PSF model on a grid across the CCD
psf_model = calexp.getPsf()
nx_grid, ny_grid = 15, 15
bbox = calexp.getBBox()

grid_x = np.linspace(bbox.getMinX() + 100, bbox.getMaxX() - 100, nx_grid)
grid_y = np.linspace(bbox.getMinY() + 100, bbox.getMaxY() - 100, ny_grid)

psf_e1_grid = []
psf_e2_grid = []
psf_fwhm_grid = []
gx, gy = [], []

for xi in grid_x:
    for yi in grid_y:
        try:
            pos = geom.Point2D(xi, yi)
            shape = psf_model.computeShape(pos)
            psf_e1_grid.append(shape.getE1())
            psf_e2_grid.append(shape.getE2())
            psf_fwhm_grid.append(shape.getDeterminantRadius() * 2.355)
            gx.append(xi)
            gy.append(yi)
        except Exception:
            pass

gx = np.array(gx)
gy = np.array(gy)
psf_e1_grid = np.array(psf_e1_grid)
psf_e2_grid = np.array(psf_e2_grid)
psf_fwhm_grid = np.array(psf_fwhm_grid)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PSF FWHM variation
ax = axes[0]
sc = ax.scatter(gx, gy, c=psf_fwhm_grid * 0.168,  # convert to arcsec
                cmap='viridis', s=60, edgecolors='gray', linewidths=0.3)
plt.colorbar(sc, ax=ax, label='FWHM (arcsec)')
ax.set_xlabel('x (pixels)'); ax.set_ylabel('y (pixels)')
ax.set_title('PSF FWHM across CCD', fontsize=13)
ax.set_aspect('equal')

# PSF ellipticity whiskers
ax = axes[1]
e_mag = np.sqrt(psf_e1_grid**2 + psf_e2_grid**2)
scale_whisker = 100
ax.quiver(gx, gy,
          psf_e1_grid * scale_whisker, psf_e2_grid * scale_whisker,
          e_mag, cmap='hot_r', scale=5,
          headwidth=0, headlength=0, headaxislength=0, linewidth=1.5)
ax.set_xlabel('x (pixels)'); ax.set_ylabel('y (pixels)')
ax.set_title('PSF ellipticity pattern', fontsize=13)
ax.set_aspect('equal')

plt.suptitle(f'PSF Model (visit={visit}, det={detector})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../figures/psf_model_pattern.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"PSF FWHM range: {psf_fwhm_grid.min()*0.168:.3f}–{psf_fwhm_grid.max()*0.168:.3f} arcsec")
print(f"PSF ellipticity range: |e| = {e_mag.min():.4f}–{e_mag.max():.4f}")

## 5. Visualize Actual PSF Images Across the CCD

In [ ]:
# Render PSF model at a 4x4 grid of positions
fig, axes = plt.subplots(4, 4, figsize=(12, 12))

sample_x = np.linspace(bbox.getMinX() + 200, bbox.getMaxX() - 200, 4)
sample_y = np.linspace(bbox.getMinY() + 200, bbox.getMaxY() - 200, 4)

for iy, yi in enumerate(sample_y):
    for ix, xi in enumerate(sample_x):
        pos = geom.Point2D(xi, yi)
        psf_im = psf_model.computeImage(pos).array
        ax = axes[iy, ix]
        ax.imshow(psf_im, cmap='inferno', origin='lower')
        shape = psf_model.computeShape(pos)
        fwhm = shape.getDeterminantRadius() * 2.355 * 0.168
        ax.set_title(f'({int(xi)},{int(yi)})\nFWHM={fwhm:.2f}"',
                     fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('PSF Model Images at Different CCD Positions',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../figures/psf_images_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Star-PSF Residual Images

The most direct check: subtract the PSF model from actual star images.

In [ ]:
# Pick a few bright, unsaturated PSF stars
bright_stars = psf_stars.copy(deep=True)
fluxes = bright_stars['base_PsfFlux_instFlux']
order = np.argsort(-fluxes)  # brightest first

n_show = 5
stamp_half = 15  # pixels

fig, axes = plt.subplots(3, n_show, figsize=(3 * n_show, 9))

for i in range(n_show):
    idx = order[i]
    star = bright_stars[idx]
    cx = int(star['base_SdssCentroid_x'])
    cy = int(star['base_SdssCentroid_y'])

    # Cut out star stamp from the calibrated exposure
    bbox_stamp = geom.Box2I(
        geom.Point2I(cx - stamp_half, cy - stamp_half),
        geom.Extent2I(2 * stamp_half + 1, 2 * stamp_half + 1)
    )

    try:
        star_stamp = calexp[bbox_stamp].image.array.copy()

        # Get PSF model at this position, scaled to star flux
        pos = geom.Point2D(cx, cy)
        psf_stamp = psf_model.computeImage(pos).array
        # Resize PSF stamp to match
        ph, pw = psf_stamp.shape
        sh, sw = star_stamp.shape
        if ph != sh or pw != sw:
            # Center-crop or pad
            min_h, min_w = min(ph, sh), min(pw, sw)
            psf_crop = psf_stamp[ph//2-min_h//2:ph//2+min_h//2+1,
                                 pw//2-min_w//2:pw//2+min_w//2+1]
            star_crop = star_stamp[sh//2-min_h//2:sh//2+min_h//2+1,
                                   sw//2-min_w//2:sw//2+min_w//2+1]
        else:
            psf_crop = psf_stamp
            star_crop = star_stamp

        # Scale PSF model to match star flux
        scale = np.sum(star_crop) / np.sum(psf_crop)
        psf_scaled = psf_crop * scale

        residual = star_crop - psf_scaled

        vmax = np.max(np.abs(star_crop)) * 0.5
        axes[0, i].imshow(star_crop, cmap='viridis', origin='lower')
        axes[0, i].set_title(f'Star {i+1}', fontsize=10)
        axes[1, i].imshow(psf_scaled, cmap='viridis', origin='lower')
        axes[1, i].set_title('PSF model', fontsize=10)
        axes[2, i].imshow(residual, cmap='RdBu_r', origin='lower',
                          vmin=-vmax*0.1, vmax=vmax*0.1)
        axes[2, i].set_title('Residual', fontsize=10)
    except Exception as e:
        axes[0, i].text(0.5, 0.5, str(e)[:30], transform=axes[0, i].transAxes,
                        ha='center', fontsize=8)

    for row in range(3):
        axes[row, i].set_xticks([]); axes[row, i].set_yticks([])

axes[0, 0].set_ylabel('Data', fontsize=12)
axes[1, 0].set_ylabel('PSF Model', fontsize=12)
axes[2, 0].set_ylabel('Residual', fontsize=12)

plt.suptitle('Star − PSF Model Residuals', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../figures/star_psf_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

print("Residuals should be noise-like.")
print("Structured residuals (rings, dipoles) indicate PSF model inadequacy.")

## Summary: PSF Quality Checklist

| Metric | Requirement (LSST) | What to check |
|--------|-------------------|---------------|
| $\langle \Delta T / T \rangle$ | < 0.001 | Size residual distribution |
| $\langle \Delta e_i \rangle$ | < 0.0002 | Ellipticity residual distribution |
| Spatial pattern | No coherent structure | Residual maps |
| Star-model images | Noise-like residuals | Visual inspection |
| ρ-statistics | Below survey spec | Correlation functions (advanced) |

If the PSF model passes these checks, galaxy shapes measured with
REGAUSS will have minimal PSF-induced systematics.

**Next:** [03_shear_validation.ipynb](03_shear_validation.ipynb) — Compare measured shapes to truth